In [37]:
import os
import requests

FIN_BASE_URL = "https://finopenapi.ssafy.io/ssafy/api/v1/"

In [49]:
# 요청 메세지로 managerID를 전달하여 API Key를 발급받는다.(POST, /issuedApiKey)
response = requests.post(FIN_BASE_URL + "edu/app/reIssuedApiKey",
                         json={"managerId": "qja1998@naver.com"})

response_json = response.json()
API_KEY = response_json["apiKey"]
print("API Key: ", API_KEY)

API Key:  5812515d2a634c0a9f4728d1470ec15b


In [50]:
USER_URL = FIN_BASE_URL + "member/"

response = requests.post(USER_URL,
                         json={"userId": "qja1998@naver.com",
                               "apiKey": API_KEY})


print(f"응답 코드: {response.status_code}")
print(f"응답 내용: {response.json()}")

응답 코드: 400
응답 내용: {'responseCode': 'E4002', 'responseMessage': '이미 존재하는 ID입니다.'}


In [51]:
response = requests.post(USER_URL + 'search',
                         json={"userId": "qja1998@naver.com",
                               "apiKey": API_KEY})


print(f"응답 코드: {response.status_code}")
print(f"응답 내용: {response.json()}")
user_key = response.json()['userKey']

응답 코드: 201
응답 내용: {'userId': 'qja1998@naver.com', 'userName': 'qja1998', 'institutionCode': '00100', 'userKey': '2c2b20d9-572c-45d4-911a-0f6842c02fec', 'created': '2025-03-21T17:24:45.35412+09:00', 'modified': '2025-03-21T17:24:45.354119+09:00'}


In [52]:
import json
from datetime import datetime

def generate_header(api_key, api_name, user_key=None):
    """현재 시간을 적용하여 헤더를 생성합니다."""
    now = datetime.now()
    transmission_date = now.strftime("%Y%m%d")
    transmission_time = now.strftime("%H%M%S")

    header = {
        "Header": {
            "apiName": api_name,
            "transmissionDate": transmission_date,
            "transmissionTime": transmission_time,
            "institutionCode": "00100",
            "fintechAppNo": "001",
            "apiServiceCode": api_name,
            "institutionTransactionUniqueNo": f"{transmission_date}{transmission_time}999999",
            "apiKey": api_key,
            "userKey": user_key
        }
    }
    return header

# 헤더 생성 및 출력 예시
header = generate_header(API_KEY, 'createDemandDeposit')
print(json.dumps(header, indent=4))

{
    "Header": {
        "apiName": "createDemandDeposit",
        "transmissionDate": "20250321",
        "transmissionTime": "172947",
        "institutionCode": "00100",
        "fintechAppNo": "001",
        "apiServiceCode": "createDemandDeposit",
        "institutionTransactionUniqueNo": "20250321172947999999",
        "apiKey": "5812515d2a634c0a9f4728d1470ec15b",
        "userKey": null
    }
}


In [59]:
# 예금 
# 통장 등록
import json

ACCOUNT_URL = FIN_BASE_URL + "edu/demandDeposit/"


CREATE_ACCOUNT_URL = ACCOUNT_URL +"createDemandDeposit"

payload = {
    "Header": generate_header(API_KEY, 'createDemandDeposit', user_key=user_key)['Header'],
    "bankCode": "001",
    "accountName": "테스트통장",
    "accountDescription": "테스트통장입니다",
}

create_response = requests.post(
        CREATE_ACCOUNT_URL,
        headers={"Content-Type": "application/json"},
        data=json.dumps(payload)
    )

print(f"응답 코드: {create_response.status_code}")
print(f"응답 내용: {create_response.json()}")
account_no = create_response.json()['REC']['accountTypeUniqueNo']

응답 코드: 201
응답 내용: {'Header': {'responseCode': 'H0000', 'responseMessage': '정상처리 되었습니다.', 'apiName': 'createDemandDeposit', 'transmissionDate': '20250321', 'transmissionTime': '173355', 'institutionCode': '00100', 'apiKey': '5812515d2a634c0a9f4728d1470ec15b', 'apiServiceCode': 'createDemandDeposit', 'institutionTransactionUniqueNo': '20250321173355999999'}, 'REC': {'accountTypeUniqueNo': '001-1-8f99bfd1279240', 'bankCode': '001', 'bankName': '한국은행', 'accountTypeCode': '1', 'accountTypeName': '수시입출금', 'accountName': '테스트통장', 'accountDescription': '테스트통장입니다', 'accountType': 'DOMESTIC'}}


In [ ]:
# 결제 프로세스
payload = {
    "Header": generate_header(API_KEY, 'inquireDemandDepositAccount', user_key=user_key)['Header'],
}

account_response = requests.get(
    ACCOUNT_URL + "inquireDemandDepositAccount",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload)
    )

account_response_json = account_response.json()
account_no = account_response_json['accountNo']
print(account_response_json['accountNo'])

# 1. 결제 요청
PAYMENT_URL = ACCOUNT_URL + "updateDemandDepositAccountWithdrawal"

payload = {
    "Header": generate_header(API_KEY, 'updateDemandDepositAccountWithdrawal', user_key=user_key)['Header'],
    "transationBalance": "10000",
    "transactionSummary": "테스트 결제",
    "accountNo": account_no
}

payment_response = requests.post(
    PAYMENT_URL,
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload)
    )

print(f"응답 코드: {payment_response.status_code}")
print(f"응답 내용: {payment_response.json()}")
# 2. 결제 승인
# 3. 상호명과 주소를 메모에 추가

KeyError: 'accountNo'

In [65]:
payload = {
    "Header": generate_header(API_KEY, 'inquireDemandDepositAccount', user_key=user_key)['Header'],
}

account_response = requests.get(
    ACCOUNT_URL + "inquireDemandDepositAccount",
    headers={"Content-Type": "application/json"},
    data=json.dumps(payload)
    )

account_response_json = account_response.json()

In [66]:
print(f"응답 코드: {payment_response.status_code}")
print(f"응답 내용: {payment_response.json()}")

응답 코드: 400
응답 내용: {'responseCode': 'A1003', 'responseMessage': '계좌번호가 유효하지 않습니다.'}


In [56]:
print("https://finopenapi.ssafy.io/ssafy/api/v1/edu/demandDeposit/updateDemandDepositAccountWithdrawal")
print(PAYMENT_URL)

https://finopenapi.ssafy.io/ssafy/api/v1/edu/demandDeposit/updateDemandDepositAccountWithdrawal
https://finopenapi.ssafy.io/ssafy/api/v1/edu/demandDeposit/updateDemandDepositAccountWithdrawal
